In [1]:
# =============================================================================
# Candlestick Classification — FOCUSED ViT Pipeline
# Assignment Task 4: Model Implementation & Hyperparameter Tuning
# =============================================================================
#
# ROOT CAUSE DIAGNOSIS (why all 3 models got ~41-47%):
# ─────────────────────────────────────────────────────
# Your models weren't failing because of architecture. They were failing because:
#
# 1. WRONG NORMALIZATION for ViT from HuggingFace
#    torchvision ViT uses ImageNet stats: mean=[0.485,0.456,0.406]
#    google/vit-base-patch16-224-in21k uses: mean=[0.5, 0.5, 0.5]
#    → Mismatched normalization = garbage input to the model
#
# 2. TOO FEW LAYERS UNFROZEN
#    Unfreezing only 1-3 encoder blocks on 18K images = model can't adapt
#    → Need full fine-tuning with layer-wise LR decay
#
# 3. StepLR drops LR by 10x every 3 epochs = kills learning too early
#    → Use Cosine with warmup instead
#
# 4. optimizer was using ALL parameters (frozen ones too) → wasted compute
#    → Filter only trainable params
#
# 5. No gradient clipping → ViT training instability
#
# STRATEGY FOR THIS FILE:
# ─────────────────────────
# Use google/vit-base-patch16-224-in21k (ImageNet-21k, 14M images)
# This is the BEST general ViT backbone available publicly.
# No candlestick-specific pretrained ViT exists publicly.
# We compensate with: correct preprocessing + full fine-tuning + LLRD
# =============================================================================

# ──────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies
# ──────────────────────────────────────────────────────────────────────────────
# !pip install transformers timm -q

# ──────────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports & Environment
# ──────────────────────────────────────────────────────────────────────────────
import os, warnings, logging
from pathlib import Path
from typing import Literal

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import datasets, transforms
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# HuggingFace ViT — MUCH better than torchvision for fine-tuning control
from transformers import ViTForImageClassification, ViTConfig

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(H:%M:%S) | %(message)s")
logger = logging.getLogger(__name__)

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device : {DEVICE}")
if torch.cuda.is_available():
    logger.info(f"GPU    : {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


# ──────────────────────────────────────────────────────────────────────────────
# CELL 3 — Configuration
# ──────────────────────────────────────────────────────────────────────────────
class Config:
    DATA_DIR        = Path("/kaggle/input/datasets/romromkankane/stock-dataset/stock_dataset/dataset")
    CHECKPOINT_DIR  = Path("/kaggle/working/checkpoints")

    # ── ViT backbone choices (ordered best → good for this task):
    # "google/vit-base-patch16-224-in21k"  ← BEST: pretrained on 14M images, no classification head
    # "google/vit-base-patch32-224-in21k"  ← patch32: larger patches, faster, slightly worse
    # "google/vit-large-patch16-224-in21k" ← larger but needs more VRAM & data
    VIT_CHECKPOINT  = "google/vit-base-patch16-224-in21k"

    BATCH_SIZE      = 32
    EPOCHS          = 20
    WARMUP_EPOCHS   = 2

    # Layer-wise LR decay: head gets full LR, earlier layers get progressively smaller
    HEAD_LR         = 2e-4     # classification head
    BACKBONE_LR     = 5e-5     # ViT encoder (lower = don't destroy pretrained features)
    WEIGHT_DECAY    = 1e-4
    LABEL_SMOOTHING = 0.1

    PATIENCE        = 14

    NUM_WORKERS     = 4
    PIN_MEMORY      = True

    # ViT-in21k uses [0.5, 0.5, 0.5] NOT ImageNet [0.485, 0.456, 0.406]
    # THIS WAS ONE OF THE ROOT CAUSES OF YOUR ~45% ACCURACY
    MEAN = [0.5, 0.5, 0.5]
    STD  = [0.5, 0.5, 0.5]

cfg = Config()
cfg.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


# ──────────────────────────────────────────────────────────────────────────────
# CELL 4 — Data Pipeline
# ──────────────────────────────────────────────────────────────────────────────
def get_transforms():
    """
    Candlestick-appropriate augmentations.

    Key decisions:
    - Mild augmentations only: candlestick charts are structured synthetic images.
      Heavy augmentations (elastic transforms, large crops) destroy the exact
      spatial price structure the model needs to learn.
    - NO vertical flip: price direction (up/down body position) is semantically
      meaningful — flipping it inverts the signal.
    - RandomHorizontalFlip: time-mirroring is structurally neutral
    - Mild ColorJitter: normalize brightness differences between data sources
    - RandomAffine with small params: account for slight scan/export artifacts
    """
    train_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.4),
        transforms.RandomRotation(degrees=5),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03)),
        transforms.ToTensor(),
        transforms.Normalize(mean=cfg.MEAN, std=cfg.STD),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=cfg.MEAN, std=cfg.STD),
    ])
    return train_tf, val_tf


def get_dataloaders():
    train_tf, val_tf = get_transforms()

    train_set = datasets.ImageFolder(cfg.DATA_DIR / "train", transform=train_tf)
    val_set   = datasets.ImageFolder(cfg.DATA_DIR / "val",   transform=val_tf)
    test_set  = datasets.ImageFolder(cfg.DATA_DIR / "test",  transform=val_tf)

    lkw = dict(
        batch_size  = cfg.BATCH_SIZE,
        num_workers = cfg.NUM_WORKERS,
        pin_memory  = cfg.PIN_MEMORY,
        persistent_workers = (cfg.NUM_WORKERS > 0),
    )
    train_loader = DataLoader(train_set, shuffle=True,  **lkw)
    val_loader   = DataLoader(val_set,   shuffle=False, **lkw)
    test_loader  = DataLoader(test_set,  shuffle=False, **lkw)

    class_names = train_set.classes
    counts      = np.bincount(train_set.targets)

    logger.info(f"Classes      : {class_names}")
    logger.info(f"Sizes        : Train={len(train_set)} | Val={len(val_set)} | Test={len(test_set)}")
    logger.info(f"Distribution : {dict(zip(class_names, counts.tolist()))}")

    return train_loader, val_loader, test_loader, class_names, train_set


# ──────────────────────────────────────────────────────────────────────────────
# CELL 5 — ViT Model (HuggingFace — Better than torchvision for this task)
# ──────────────────────────────────────────────────────────────────────────────
def build_vit(num_classes: int, class_names: list) -> nn.Module:
    """
    WHY google/vit-base-patch16-224-in21k INSTEAD OF torchvision vit_b_16?
    ────────────────────────────────────────────────────────────────────────
    torchvision vit_b_16(IMAGENET1K_V1):
      - Trained on ImageNet-1k (1.28M images, 1000 classes)
      - Already has a classification head for 1000 classes we must replace
      - Uses ImageNet normalization stats [0.485, 0.456, 0.406]

    google/vit-base-patch16-224-in21k:
      - Trained on ImageNet-21k (14M images, 21,843 classes) ← 11× more data
      - Richer, more general visual representations
      - No classification head → we attach our own cleanly
      - Uses [0.5, 0.5, 0.5] normalization
      - This is the checkpoint all major ViT fine-tuning papers use

    WHY NO CANDLESTICK-SPECIFIC PRETRAINED ViT EXISTS:
    ────────────────────────────────────────────────────
    Searched HuggingFace Hub thoroughly. No publicly available ViT model
    pretrained specifically on candlestick or financial chart images exists.
    Papers like Byun et al. (2025) train their own from scratch on
    large proprietary datasets. Our best option is the strongest available
    general backbone + proper fine-tuning strategy.

    FINE-TUNING STRATEGY — Layer-wise LR Decay (LLRD):
    ────────────────────────────────────────────────────
    - Head LR        : 2e-4   (newly initialized, needs to learn fast)
    - Encoder blocks : 5e-5   (pretrained, should adapt slowly)
    - Earlier layers : lower  (low-level features are universal, barely touch)
    This prevents destroying pretrained features while allowing task adaptation.
    """
    id2label = {i: c for i, c in enumerate(class_names)}
    label2id = {c: i for i, c in enumerate(class_names)}

    model = ViTForImageClassification.from_pretrained(
        cfg.VIT_CHECKPOINT,
        num_labels          = num_classes,
        id2label            = id2label,
        label2id            = label2id,
        ignore_mismatched_sizes = True,   # replaces the pretrained head cleanly
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    logger.info(f"[ViT] {cfg.VIT_CHECKPOINT}")
    logger.info(f"[ViT] Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    return model.to(DEVICE)


def get_optimizer_with_llrd(model: nn.Module) -> optim.Optimizer:
    """
    Layer-wise Learning Rate Decay (LLRD).

    Problem without LLRD:
      Using one LR for the entire model either:
      (a) too high → destroys pretrained features in early encoder layers
      (b) too low  → head and late layers don't learn fast enough

    Solution — assign different LRs per layer group:
      classifier (head) → HEAD_LR  (2e-4)  learn fast
      encoder.layer[11] → BACKBONE_LR      learn normally
      encoder.layer[10] → BACKBONE_LR * 0.9
      encoder.layer[9]  → BACKBONE_LR * 0.8
      ...
      embeddings        → BACKBONE_LR * 0.1 barely touch

    Example LRs for 12 encoder layers:
      Head    : 2.00e-04
      Layer 11: 5.00e-05
      Layer 10: 4.50e-05
      Layer 9 : 4.05e-05
      ...
      Layer 0 : ~1.9e-06
    """
    decay  = 0.9
    params = []

    # Classification head — highest LR
    params.append({
        "params": model.classifier.parameters(),
        "lr": cfg.HEAD_LR,
    })

    # ViT encoder — layer-wise decay (later layers → higher LR)
    num_layers = len(model.vit.encoder.layer)
    for i, block in enumerate(reversed(model.vit.encoder.layer)):
        layer_lr = cfg.BACKBONE_LR * (decay ** i)
        params.append({"params": block.parameters(), "lr": layer_lr})

    # Embeddings — very low LR (patch embed + position embed are universal)
    params.append({
        "params": model.vit.embeddings.parameters(),
        "lr": cfg.BACKBONE_LR * (decay ** num_layers),
    })

    # LayerNorm before head
    params.append({
        "params": model.vit.layernorm.parameters(),
        "lr": cfg.BACKBONE_LR,
    })

    optimizer = optim.AdamW(params, weight_decay=cfg.WEIGHT_DECAY, betas=(0.9, 0.999))

    # Log LR distribution
    lrs = [g["lr"] for g in params]
    logger.info(f"[LLRD] LR range: {min(lrs):.2e} → {max(lrs):.2e}")
    return optimizer


# ──────────────────────────────────────────────────────────────────────────────
# CELL 6 — Training Utilities
# ──────────────────────────────────────────────────────────────────────────────
def get_class_weights(dataset) -> torch.Tensor:
    counts  = np.bincount(dataset.targets).astype(float)
    weights = 1.0 / counts
    weights /= weights.sum()
    logger.info(f"Class weights: {weights.round(4)}")
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    """
    Mixed precision training (torch.cuda.amp).

    Why AMP matters here:
    - ViT-B/16 has 86M parameters — FP32 training on T4 is slow
    - AMP uses FP16 for forward/backward → ~1.7× faster
    - GradScaler handles FP16 gradient underflow automatically
    - Gradient clipping (max_norm=1.0) critical for ViT stability
    """
    model.train()
    running_loss = correct = total = 0

    for batch in tqdm(loader, desc="Train", leave=False):
        # HuggingFace models take pixel_values, but ImageFolder returns (image, label)
        images, labels = batch
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(pixel_values=images)
            logits  = outputs.logits
            loss    = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted  = logits.max(1)
        total        += labels.size(0)
        correct      += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = correct = total = 0

    for images, labels in tqdm(loader, desc="Eval ", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast():
            outputs = model(pixel_values=images)
            logits  = outputs.logits
            loss    = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted  = logits.max(1)
        total        += labels.size(0)
        correct      += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total


# ──────────────────────────────────────────────────────────────────────────────
# CELL 7 — Main Training Loop
# ──────────────────────────────────────────────────────────────────────────────
def train(model, train_loader, val_loader, class_names, train_dataset):
    """
    Full training pipeline.

    Scheduler: Warmup (2 epochs) → CosineAnnealingLR
    ────────────────────────────────────────────────
    Why NOT StepLR (what you had before):
      StepLR drops LR by 10× every 3 epochs:
        Epoch 1-3: LR = 1e-4
        Epoch 4-6: LR = 1e-5  ← model stops learning useful patterns
        Epoch 7-9: LR = 1e-6  ← effectively frozen

    Why CosineAnnealingLR with warmup:
      - Warmup: slowly ramp LR over first 2 epochs
        → prevents ViT head from blowing up on first batches
      - Cosine: smoothly decays from peak LR to eta_min=1e-6
        → model keeps learning throughout, just more carefully over time
    """
    class_weights = get_class_weights(train_dataset)
    criterion = nn.CrossEntropyLoss(
        weight         = class_weights,
        label_smoothing = cfg.LABEL_SMOOTHING,
    )

    optimizer = get_optimizer_with_llrd(model)

    warmup   = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=cfg.WARMUP_EPOCHS)
    cosine   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS - cfg.WARMUP_EPOCHS, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[cfg.WARMUP_EPOCHS])

    scaler          = GradScaler()
    best_val_acc    = 0.0
    patience_ctr    = 0
    checkpoint_path = cfg.CHECKPOINT_DIR / "best_vit.pth"

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}

    logger.info(f"{'='*65}")
    logger.info(f"  Training: {cfg.VIT_CHECKPOINT}")
    logger.info(f"  Epochs={cfg.EPOCHS} | Warmup={cfg.WARMUP_EPOCHS} | Batch={cfg.BATCH_SIZE}")
    logger.info(f"  Head LR={cfg.HEAD_LR:.1e} | Backbone LR={cfg.BACKBONE_LR:.1e} | LLRD=True")
    logger.info(f"{'='*65}")

    for epoch in range(1, cfg.EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion)
        scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        for key, val in zip(
            ["train_loss", "train_acc", "val_loss", "val_acc", "lr"],
            [train_loss, train_acc, val_loss, val_acc, lr_now]
        ):
            history[key].append(val)

        logger.info(
            f"[{epoch:02d}/{cfg.EPOCHS}] "
            f"Train loss={train_loss:.4f} acc={train_acc:.2f}%  |  "
            f"Val loss={val_loss:.4f} acc={val_acc:.2f}%  |  "
            f"LR={lr_now:.2e}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_ctr = 0
            torch.save({
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "val_acc":     val_acc,
                "class_names": class_names,
                "checkpoint":  cfg.VIT_CHECKPOINT,
            }, checkpoint_path)
            logger.info(f"  ✓ Best model saved — val_acc={best_val_acc:.2f}%")
        else:
            patience_ctr += 1
            if patience_ctr >= cfg.PATIENCE:
                logger.info(f"  Early stopping at epoch {epoch}")
                break

    logger.info(f"Training complete. Best Val Acc: {best_val_acc:.2f}%")
    return history, checkpoint_path


# ──────────────────────────────────────────────────────────────────────────────
# CELL 8 — Evaluation
# ──────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_test_set(model, loader, class_names, checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    logger.info(f"Loaded checkpoint from epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.2f}%)")

    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc="Test"):
        images = images.to(DEVICE, non_blocking=True)
        with autocast():
            out = model(pixel_values=images)
        _, preds = out.logits.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

    print("\n" + "="*60)
    print("  TEST SET RESULTS — ViT-B/16 (in21k)")
    print("="*60)
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

    _plot_confusion_matrix(all_labels, all_preds, class_names)
    return np.array(all_preds), np.array(all_labels)


def _plot_confusion_matrix(labels, preds, class_names):
    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names, yticklabels=class_names,
                cmap="Blues", linewidths=0.5, ax=axes[0])
    axes[0].set_title("Confusion Matrix (counts)", fontweight="bold")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

    sns.heatmap(cm_norm, annot=True, fmt=".1%",
                xticklabels=class_names, yticklabels=class_names,
                cmap="RdYlGn", linewidths=0.5, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title("Confusion Matrix (per-class %)", fontweight="bold")
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

    fig.suptitle("ViT-B/16 (ImageNet-21k fine-tuned) — Test Evaluation",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(cfg.CHECKPOINT_DIR / "confusion_matrix_vit.png", dpi=150)
    plt.show()


def plot_training_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss
    axes[0].plot(history["train_loss"], label="Train", lw=2)
    axes[0].plot(history["val_loss"],   label="Val",   lw=2)
    axes[0].set_title("Loss Curve", fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(history["train_acc"], label="Train", lw=2)
    axes[1].plot(history["val_acc"],   label="Val",   lw=2)
    axes[1].axhline(y=33.3, color="red", ls="--", alpha=0.5, label="Random (33.3%)")
    axes[1].set_title("Accuracy Curve", fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    # LR
    axes[2].plot(history["lr"], lw=2, color="darkorange")
    axes[2].axvline(x=cfg.WARMUP_EPOCHS, color="gray", ls="--", alpha=0.6, label=f"End warmup (ep {cfg.WARMUP_EPOCHS})")
    axes[2].set_title("LR Schedule (Warmup + Cosine)", fontweight="bold")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("LR")
    axes[2].set_yscale("log"); axes[2].legend(); axes[2].grid(alpha=0.3)

    fig.suptitle("ViT-B/16 Training History", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(cfg.CHECKPOINT_DIR / "training_history_vit.png", dpi=150)
    plt.show()


# ──────────────────────────────────────────────────────────────────────────────
# CELL 9 — Entrypoint
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Step 1 — Data
    train_loader, val_loader, test_loader, class_names, train_dataset = get_dataloaders()

    # Step 2 — Model
    model = build_vit(num_classes=len(class_names), class_names=class_names)

    # Step 3 — Train
    history, checkpoint_path = train(model, train_loader, val_loader, class_names, train_dataset)

    # Step 4 — Visualize training
    plot_training_history(history)

    # Step 5 — Final test evaluation
    preds, labels = evaluate_test_set(model, test_loader, class_names, checkpoint_path)

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 464, in format
    return self._format(record)
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 460, in _format
    return self._fmt % values
           ~~~~~~~~~~^~~~~~~~
KeyError: 'H:%M:%S'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 706, in format
    s = self.formatMessage(record)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 675, in formatMessage
    return self._style.format(record)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
 

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 464, in format
    return self._format(record)
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 460, in _format
    return self._fmt % values
           ~~~~~~~~~~^~~~~~~~
KeyError: 'H:%M:%S'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 706, in format
    s = self.formatMessage(record)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 675, in formatMessage
    return self._style.format(record)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
 

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

Train:   0%|          | 0/413 [00:00<?, ?it/s]

Eval :   0%|          | 0/118 [00:00<?, ?it/s]

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 464, in format
    return self._format(record)
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 460, in _format
    return self._fmt % values
           ~~~~~~~~~~^~~~~~~~
KeyError: 'H:%M:%S'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 706, in format
    s = self.formatMessage(record)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 675, in formatMessage
    return self._style.format(record)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
 

Train:   0%|          | 0/413 [00:00<?, ?it/s]

KeyboardInterrupt: 